# Klasterovanje gena — vizualizacija algoritama korak po korak

Prateća sveska uz glavnu (`klasterovanje_u_bioinformatici.ipynb`): interaktivni prikaz rada
algoritama **korak po korak** na istom skupu od deset gena kvasca.

**Kako se koristi:** pokreni ćelije redom od vrha (definicije i podaci se učitaju jednom), a onda
na svakoj ćeliji svaki `Shift+Enter` prikazuje sledeći korak (grafik + tekstualni opis).
Za ponovni početak koristi `reset=True`, npr. `next_kmeans(data, km_history, reset=True)`.


In [ ]:
# osnovni uvozi (koje viz-kod koristi)
import math
import random


In [ ]:
# Stvarni skup od 10 gena kvasca: 10x7 podmatrica DeRisijeve matrice log2 fold change
# u sedam vremenskih tacaka oko diauksicne promene.
# Preuzeto iz Compeau & Pevzner, "Bioinformatics Algorithms" (Slika 8.3).
genes = [
    # Geni regulisani na gore (respiracija)
    [-0.43, -0.73, -0.06, -0.11, -0.16, 3.47, 2.64],   # YGR043C
    [0.11, 0.43, 0.45, 1.89, 2.00, 3.32, 2.56],        # YLR258W (glikogen sintaza)
    [-0.19, -0.15, 0.03, 0.27, 0.54, 3.64, 2.74],      # YKL026C
    # Geni regulisani na dole (fermentacija)
    [0.12, -0.23, -0.24, -1.16, -1.40, -2.67, -3.00],  # YMR290C
    [0.09, -0.28, -0.15, -1.18, -1.59, -2.96, -3.08],  # YPL012W
    [-0.16, -0.04, -0.07, -1.26, -1.20, -2.82, -3.13], # YNL141W
    # Nepromenjeni geni (housekeeping)
    [0.14, 0.03, -0.06, 0.07, -0.01, -0.06, -0.01],    # YLR361C
    [-0.10, -0.14, -0.03, -0.06, -0.07, -0.14, -0.04], # YNR065C
    [-0.28, -0.23, -0.19, -0.19, -0.32, -0.18, -0.18], # YJL028W
    [0.15, 0.15, 0.17, 0.09, 0.07, 0.09, 0.07],        # YPR055W
]

gene_names = ["YGR043C", "YLR258W", "YKL026C",
              "YMR290C", "YPL012W", "YNL141W",
              "YLR361C", "YNR065C", "YJL028W", "YPR055W"]

print(f"Broj gena: {len(genes)}")
print(f"Broj vremenskih tacaka: {len(genes[0])}")
print()
print("Matrica ekspresije (log2 fold change):")
print(f"{'Gen':<10} {'t1':>6} {'t2':>6} {'t3':>6} {'t4':>6} {'t5':>6} {'t6':>6} {'t7':>6}")
print("-" * 62)
for i, gene in enumerate(genes):
    values_str = " ".join(f"{v:>6.2f}" for v in gene)
    print(f"{gene_names[i]:<10} {values_str}")


## Korak po korak: vizualizacija algoritama

Tri algoritma (K-means, FarthestFirstTraversal, DBSCAN) na 2D igračkim podacima. Slika se osvežava na **istoj osi**.

**Kontrola:** svaki **Shift+Enter** na ćeliji pomera algoritam za jedan korak. Kad se stigne do kraja, vraća se na početak. Reset: pozovi funkciju sa `reset=True`.

### Zajedničke funkcije (pokreni jednom)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

COLORS = ["tab:blue", "tab:orange", "tab:green",
          "tab:red", "tab:purple", "tab:brown"]

def euclidean(a, b):
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

def centroid_of(points):
    n = len(points)
    m = len(points[0])
    return [sum(p[d] for p in points) / n for d in range(m)]

def axis_bounds(data, margin=0.6):
    xs = [p[0] for p in data]
    ys = [p[1] for p in data]
    return (min(xs) - margin, max(xs) + margin,
            min(ys) - margin, max(ys) + margin)

def assign(data, centers):
    """Dodela svake tacke najblizem centru (k-sredina, brute-force)."""
    labels = []
    for p in data:
        d = [euclidean(p, c) for c in centers]
        labels.append(d.index(min(d)))
    return labels

def neighbors(data, i, eps):
    """Indeksi tacaka u eps-okolini tacke i (DBSCAN)."""
    return [j for j in range(len(data))
            if j != i and euclidean(data[i], data[j]) <= eps]

_STEP_STATE = {}

def _print_step(title, idx, total, opis):
    """Pregledan tekstualni ispis jednog koraka (zajednicki za sve algoritme)."""
    done = idx + 1
    bar_len = 24
    fill = round(bar_len * done / total)
    bar = "[" + "#" * fill + "-" * (bar_len - fill) + "]"
    pct = round(100 * done / total)
    head = f"  {title}"
    cnt = f"korak {done} / {total}  "
    pad = max(1, 60 - len(head) - len(cnt))
    print("=" * 60)
    print(head + " " * pad + cnt)
    print("-" * 60)
    print(f"  {bar}  {pct}%")
    print(f"  {opis}")
    print("-" * 60)
    if done == total:
        print("  KRAJ - sledeci Shift+Enter krece ispocetka (reset=True rucno)")
    else:
        print("  Shift+Enter  ->  sledeci korak")
    print("=" * 60)

def _step(key, data, history, draw_fn, title, reset):
    if reset or key not in _STEP_STATE:
        _STEP_STATE[key] = -1
    _STEP_STATE[key] = (_STEP_STATE[key] + 1) % len(history)
    idx = _STEP_STATE[key]
    x0, x1, y0, y1 = axis_bounds(data)
    fig, ax = plt.subplots(figsize=(6, 6))
    draw_fn(ax, data, history, idx, title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    fig.tight_layout()
    plt.show()
    _print_step(title, idx, len(history), history[idx][-1])

def _step_soft(key, data, history, title, reset):
    """Stepper specifican za Soft K-means: levo tacke, desno HiddenMatrix R."""
    if reset or key not in _STEP_STATE:
        _STEP_STATE[key] = -1
    _STEP_STATE[key] = (_STEP_STATE[key] + 1) % len(history)
    idx = _STEP_STATE[key]
    centers, R, opis = history[idx]
    k = len(centers); n = len(data)
    fig, (axL, axR) = plt.subplots(
        1, 2, figsize=(11, 5.2), gridspec_kw={"width_ratios": [1.0, 1.15]})
    # Levo: tacke obojene po pripadnosti (postojeci crtez)
    draw_soft_kmeans_step(axL, data, history, idx, title)
    axL.set_aspect("equal"); axL.grid(True, alpha=0.3)
    x0, x1, y0, y1 = axis_bounds(data)
    axL.set_xlim(x0, x1); axL.set_ylim(y0, y1)
    # Desno: HiddenMatrix = matrica pripadnosti R[i][j]
    if R is None:
        axR.text(0.5, 0.5, "HiddenMatrix se racuna\nu E-koraku\n(pritisni Shift+Enter)",
                 ha="center", va="center", fontsize=12)
        axR.axis("off")
    else:
        im = axR.imshow(R, aspect="auto", cmap="viridis", vmin=0, vmax=1)
        axR.set_yticks(range(k))
        axR.set_yticklabels([f"klaster {i}" for i in range(k)])
        axR.set_xlabel(f"tacke (geni)  j = 0 .. {n - 1}")
        axR.set_title("HiddenMatrix  R[i][j] = P(tacka j -> klaster i)", fontsize=10)
        fig.colorbar(im, ax=axR, fraction=0.046, pad=0.04, label="pripadnost")
    fig.tight_layout()
    plt.show()
    _print_step(title, idx, len(history), opis)
    # Numericki uzorak: nekoliko kolona HiddenMatrix (svaka kolona se sabira na 1)
    if R is not None:
        for j in (0, n // 2, n - 1):
            vals = "   ".join(f"k{i}={R[i][j]:.2f}" for i in range(k))
            tot = sum(R[i][j] for i in range(k))
            print(f"  tacka {j:>2}:  {vals}   (zbir={tot:.2f})")
        print("=" * 60)

def draw_clusters_2d(ax, data, history, idx, title):
    """Zajednicki crtez: tacke obojene po klasteru. Koristi se i za DIANA."""
    clusters, opis = history[idx]
    for ci, cluster in enumerate(clusters):
        for i in cluster:
            ax.scatter([data[i][0]], [data[i][1]],
                       c=COLORS[ci % len(COLORS)], s=95, alpha=0.9, edgecolors="black", linewidths=0.5)
    ax.set_title(f"{title} (korak {idx+1}/{len(history)})\n{opis}", fontsize=11)

data = [[sum(v[3:]) / 4.0, max(v) - min(v)] for v in genes]
print('Zajednicke funkcije ucitane. Pokreni celiju ispod svakog algoritma.')

### FarthestFirstTraversal

Svaki sledeci centar je tacka **najudaljenija** od svih dosad izabranih (max-min rastojanje). Vidi se kako algoritam "skace" iz klastera u klaster da pokrije ceo prostor.

**Opis:**
- **Sive tacke**: sve ulazne tacke (27 u 2D)
- **Crveni X sa crnim obrubom**: izabrani centri
- **Crveni krug bez ispune** oko poslednjeg X: novododati centar u trenutnom koraku
- **Naslov panela**: broj koraka i indeks tacke koja je dodata kao novi centar
- **Sta gledati**: kako se centri "razlete" po prostoru - nikad se ne nađu blizu jedan drugom, jer pravilo max-min biraju upravo najudaljenije

In [ ]:
def fft_steps(data, k, first=0):
    """FarthestFirstTraversal; vraca [(centers, opis), ...]."""
    centers = [list(data[first])]
    history = [([list(c) for c in centers],
                f"Korak 0: prvi centar (tacka {first})")]
    for s in range(1, k):
        d_min = [min(euclidean(p, c) for c in centers) for p in data]
        idx = d_min.index(max(d_min))
        centers.append(list(data[idx]))
        history.append(([list(c) for c in centers],
                        f"Korak {s}: dodat centar "
                        f"(tacka {idx}, najudaljenija od dosadasnjih)"))
    return history

def draw_fft_step(ax, data, history, idx, title):
    centers, opis = history[idx]
    xs_all, ys_all = zip(*data)
    ax.scatter(xs_all, ys_all, c="lightgray", s=95, edgecolors="black", linewidths=0.5)
    cxs, cys = zip(*centers)
    ax.scatter(cxs, cys, c="tab:red", marker="X", s=240,
               edgecolors="black", linewidths=1.5)
    ax.scatter([centers[-1][0]], [centers[-1][1]],
               facecolors="none", edgecolors="tab:red",
               s=500, linewidths=2)
    ax.set_title(f"{title} (korak {idx+1}/{len(history)})\n{opis}",
                 fontsize=11)

def next_fft(data, history, title="FarthestFirstTraversal", reset=False):
    _step("fft", data, history, draw_fft_step, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_fft(data, fft_history, reset=True)
fft_history = fft_steps(data, k=3)
next_fft(data, fft_history)


### K-means (Lloyd)

U svakom koraku centri se pomere u centar gravitacije svog trenutnog klastera, pa se sve tacke preraspodeluju najblizem centru. Vidi se kako centri "klize" ka stvarnim klasterima i kako se broj tacaka koje menjaju klaster smanjuje do nule (konvergencija).

**Opis:**
- **Tacke** (kruzici, 27 ukupno): pozicija = koordinate u 2D ravni; **boja** = trenutni najblizi centar (plava / narandzasta / zelena = klaster 1 / 2 / 3)
- **Veliki X**: centri klastera, obojeni po klasteru
- **Naslov panela**: `korak X/Y` i opis koliko je tacaka promenilo klaster u tom koraku
- **Sta gledati**: kako X-ovi "klize" iz pocetne pozicije ka centru gusto obojene grupe; konvergencija = kada nijedna tacka vise ne menja boju

In [ ]:
def kmeans_steps(data, k, seed=0, max_iter=20):
    """Lloyd K-means; vraca [(centers, labels, opis), ...]."""
    rnd = random.Random(seed)
    centers = [list(p) for p in rnd.sample(data, k)]
    history = []
    labels = assign(data, centers)
    history.append(([list(c) for c in centers], labels[:],
                    "Korak 0: inicijalni centri (slucajno izabrani)"))
    for it in range(1, max_iter + 1):
        new_centers = []
        for j in range(k):
            grp = [data[i] for i, l in enumerate(labels) if l == j]
            new_centers.append(centroid_of(grp) if grp else list(centers[j]))
        new_labels = assign(data, new_centers)
        changed = sum(1 for a, b in zip(labels, new_labels) if a != b)
        history.append(([list(c) for c in new_centers], new_labels[:],
                        f"Korak {it}: centri pomereni; "
                        f"{changed} tacaka promenilo klaster"))
        if changed == 0:
            break
        centers, labels = new_centers, new_labels
    return history

def draw_kmeans_step(ax, data, history, idx, title):
    centers, labels, opis = history[idx]
    for j in range(len(centers)):
        grp = [data[i] for i, l in enumerate(labels) if l == j]
        if grp:
            xs, ys = zip(*grp)
            ax.scatter(xs, ys, c=COLORS[j % len(COLORS)], s=95, alpha=0.9, edgecolors="black", linewidths=0.5)
    for j, c in enumerate(centers):
        ax.scatter([c[0]], [c[1]], c=COLORS[j % len(COLORS)],
                   marker="X", s=240,
                   edgecolors="black", linewidths=1.5)
    ax.set_title(f"{title} (korak {idx+1}/{len(history)})\n{opis}",
                 fontsize=11)

def next_kmeans(data, history, title="K-means (Lloyd)", reset=False):
    _step("km", data, history, draw_kmeans_step, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_kmeans(data, km_history, reset=True)
km_history = kmeans_steps(data, k=3, seed=0)
next_kmeans(data, km_history)


### Soft K-means (EM)

Verovatnosna varijanta K-means-a: svaka tacka pripada **svakom** centru sa nekom "odgovornoscu" (responsibility), $r_{ij} \propto e^{-\beta \cdot d(x_j, c_i)^2}$. Veci $\beta$ = krutije dodele.

**HiddenMatrix (matrica pripadnosti $R$):** desni panel.
- Svaka **kolona** je jedna tacka (gen), svaki **red** je jedan klaster.
- Vrednost $R[i][j]$ = verovatnoca da tacka $j$ pripada klasteru $i$; **svaka kolona se sabira na 1**.
- Svetlija celija = veca pripadnost. Kod **tvrdog** K-means-a svaka kolona bi imala tacno jednu jedinicu (0/1); kod **mekog** je verovatnoca razmazana - to je cela poenta.
- Naziv "HiddenMatrix" je iz udzbenika (Compeau & Pevzner): to je skrivena (latentna) raspodela pripadnosti koju EM procenjuje.

**Levi panel:**
- **Boja tacke**: klaster sa najvecom odgovornoscu (top kolona u $R$).
- **Providnost (alpha)**: jacina najvece pripadnosti - **pune** tacke su sigurne, **izbledele** su granicne (slicno bliske dvama centrima).
- **Veliki X**: trenutni centri.

**Sta gledati:** kako se u HiddenMatrix kolone "izostre" kroz EM iteracije (jedan red sve svetliji), a koje kolone ostaju razmazane - to su tacke izmedju klastera koje algoritam nije sigurno raspodelio.

In [ ]:
def soft_kmeans_steps(data, k, beta=2.0, seed=0, max_iter=20):
    """history = [(centers, R_or_None, opis), ...]; R[i][j] = pripadnost centra i za tacku j."""
    rnd = random.Random(seed)
    centers = [list(p) for p in rnd.sample(data, k)]
    history = [([list(c) for c in centers], None,
                "Korak 0: inicijalni centri (slucajno izabrani)")]
    n = len(data); m = len(data[0])
    for it in range(1, max_iter + 1):
        # E-step
        R = [[0.0] * n for _ in range(k)]
        for j, p in enumerate(data):
            ex = [math.exp(-beta * euclidean(p, c) ** 2) for c in centers]
            s = sum(ex)
            for i in range(k):
                R[i][j] = ex[i] / s if s > 0 else 1.0 / k
        # M-step
        new_centers = []
        for i in range(k):
            tw = sum(R[i][j] for j in range(n))
            if tw == 0:
                new_centers.append(list(centers[i]))
            else:
                new_centers.append([sum(R[i][j] * data[j][d] for j in range(n)) / tw
                                    for d in range(m)])
        diff = sum(euclidean(a, b) for a, b in zip(centers, new_centers))
        history.append(([list(c) for c in new_centers], [row[:] for row in R],
                        f"Korak {it}: EM iteracija; pomak centara = {diff:.3f}"))
        if diff < 1e-3:
            break
        centers = new_centers
    return history

def draw_soft_kmeans_step(ax, data, history, idx, title):
    centers, R, opis = history[idx]
    k = len(centers)
    if R is None:
        for p in data:
            ax.scatter([p[0]], [p[1]], c="lightgray", s=75, edgecolors="black", linewidths=0.5)
    else:
        for j, p in enumerate(data):
            resp = [R[i][j] for i in range(k)]
            top = resp.index(max(resp))
            alpha = max(0.6, resp[top])
            ax.scatter([p[0]], [p[1]], c=COLORS[top % len(COLORS)],
                       s=75, alpha=alpha, edgecolors="black", linewidths=0.5)
    for i, c in enumerate(centers):
        ax.scatter([c[0]], [c[1]], c=COLORS[i % len(COLORS)],
                   marker="X", s=150, edgecolors="black", linewidths=1.3)
    ax.set_title(f"{title} (korak {idx+1}/{len(history)})\n{opis}", fontsize=11)

def next_soft_kmeans(data, history, title="Soft K-means (EM, beta=2)", reset=False):
    _step_soft("soft", data, history, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_soft_kmeans(data, soft_history, reset=True)
soft_history = soft_kmeans_steps(data, k=3, beta=2.0, seed=0)
next_soft_kmeans(data, soft_history)


### Brute-force K-means

Iscrpno proba **sve kombinacije** $k$ centara iz podataka i bira onu sa najmanjom distorzijom. Za $n=27, k=3$ to je $\binom{27}{3}=2925$ kombinacija. Belezim svaku **novu najbolju** koju nadjem dok prolazim - broj koraka je $\approx 17$ (samo poboljsanja). Garantovano daje globalni optimum (ali sporo za velike $n$).

**Opis:**
- **Veliki X sa crnim obrubom**: trojka tacaka iz dataset-a koja se trenutno isproba kao centri (obojeni po klasteru)
- **Manje kruznice**: sve ulazne tacke, obojene po **najblizem od trenutnih kandidata** (kao da je k-means vec gotov sa tim centrima)
- **Naslov panela**: redni broj kombinacije od 2925, i nova najbolja distorzija
- **Sta gledati**: kako se X-ovi krecu po prostoru testirajuci kombinacije; svaki prikazan korak je STROGO BOLJI od prethodnog (distorzija pada); finalni izbor su tri tacke koje pokrivaju tri prirodne grupe

In [ ]:
import itertools

def brute_steps(data, k):
    n = len(data)
    best_dist = float("inf")
    history = []
    cnt = 0
    for combo in itertools.combinations(range(n), k):
        cnt += 1
        centers = [list(data[i]) for i in combo]
        dist = sum(min(euclidean(p, c) for c in centers) for p in data)
        if dist < best_dist:
            best_dist = dist
            history.append(([list(c) for c in centers],
                            f"Kombinacija {cnt}/{math.comb(n, k)}: nova najbolja, distorzija = {dist:.3f}"))
    return history

def draw_brute_step(ax, data, history, idx, title):
    centers, opis = history[idx]
    # dodeli tacke najblizem centru
    labels = []
    for p in data:
        d = [euclidean(p, c) for c in centers]
        labels.append(d.index(min(d)))
    for j in range(len(centers)):
        grp = [data[i] for i, l in enumerate(labels) if l == j]
        if grp:
            xs, ys = zip(*grp)
            ax.scatter(xs, ys, c=COLORS[j % len(COLORS)], s=95, alpha=0.9, edgecolors="black", linewidths=0.5)
    for j, c in enumerate(centers):
        ax.scatter([c[0]], [c[1]], c=COLORS[j % len(COLORS)],
                   marker="X", s=240, edgecolors="black", linewidths=1.5)
    ax.set_title(f"{title} (korak {idx+1}/{len(history)})\n{opis}", fontsize=10)

def next_brute(data, history, title="Brute-force K-means (k=3)", reset=False):
    _step("brute", data, history, draw_brute_step, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_brute(data, brute_history, reset=True)
brute_history = brute_steps(data, k=3)
next_brute(data, brute_history)


### UPGMA (hijerarhijsko klasterovanje, bottom-up)

Polazi od svake tacke kao zasebnog klastera i u svakom koraku **spaja dva najbliza klastera** (UPGMA = average linkage: rastojanje izmedju klastera = prosek svih parova). Svaki Shift+Enter spaja jedan par.

**Opis:**
- **Boja tacke**: ID trenutnog klastera kome tacka pripada (6 boja se ciklicno rotira ako klastera ima vise od 6)
- **Korak 0**: svaka tacka je svoj klaster -> 27 razlicitih grupa, boje se rotiraju
- **Sredisnji koraci**: postepeno se boje sazimaju kako se klasteri spajaju
- **Naslov panela**: korak, prosecna distanca spojenog para (`D_avg`), broj preostalih klastera
- **Sta gledati**: kako se 27 zasebnih boja postepeno sazima u 3 dominantne grupe; spajanja unutar guste grupe imaju mali `D_avg`, finalno spajanje izmedju grupa ima veliki

In [ ]:
def hierarchical_steps(data, target_k=3):
    """history = [(clusters, opis), ...]; clusters = lista lista indeksa."""
    n = len(data)
    clusters = [[i] for i in range(n)]
    history = [([c[:] for c in clusters], "Korak 0: svaka tacka je svoj klaster")]
    step = 0
    while len(clusters) > target_k:
        best_d = None; best_pair = None
        for a in range(len(clusters)):
            for b in range(a + 1, len(clusters)):
                s = 0; cnt = 0
                for i in clusters[a]:
                    for j in clusters[b]:
                        s += euclidean(data[i], data[j]); cnt += 1
                d = s / cnt
                if best_d is None or d < best_d:
                    best_d = d; best_pair = (a, b)
        a, b = best_pair
        merged = clusters[a] + clusters[b]
        clusters = [c for kk, c in enumerate(clusters) if kk not in (a, b)] + [merged]
        step += 1
        history.append(([c[:] for c in clusters],
                        f"Korak {step}: spojeno (D_avg={best_d:.2f}) -> {len(clusters)} klastera"))
    return history

def next_hierarchical(data, history, title="UPGMA", reset=False):
    _step("hier", data, history, draw_clusters_2d, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_hierarchical(data, hier_history, reset=True)
hier_history = hierarchical_steps(data, target_k=3)
next_hierarchical(data, hier_history)


### DIANA (Divisive Analysis, top-down)

Suprotno UPGMA - polazi od **jednog klastera sa svim tackama** i u svakom koraku deli klaster sa najvecim dijametrom: izdvaja najneslicniju tacku u proseku (osnivac izdvojene grupe) i preseli k njoj sve tacke koje su joj blize nego ostatku.

**Opis:**
- **Boja tacke**: ID klastera (sve tacke u istom klasteru -> ista boja)
- **Korak 0**: sve tacke iste boje (plava) - jedan klaster
- **Svaki sledeci korak**: jedan klaster se razdvoji na dva, broj boja raste za 1
- **Naslov panela**: korak, dijametar razdvojenog klastera, broj klastera nakon podele
- **Sta gledati**: koji se klaster prvi cepa (onaj sa najvecim dijametrom - obicno spaja dve fizicki udaljene gusto grupe); kako heuristika izdvojene grupe pravilno razdvaja po fizickoj udaljenosti

In [ ]:
def diana_steps(data, target_k=3):
    n = len(data)
    clusters = [list(range(n))]
    history = [([c[:] for c in clusters], "Korak 0: svi u jednom klasteru")]
    step = 0
    while len(clusters) < target_k:
        # nadji klaster sa najvecim dijametrom
        best_ci, best_diam = -1, -1
        for ci, c in enumerate(clusters):
            if len(c) < 2: continue
            diam = max(euclidean(data[i], data[j]) for i in c for j in c if i < j)
            if diam > best_diam: best_diam, best_ci = diam, ci
        if best_ci == -1: break
        cluster = clusters[best_ci]
        # splinter seed = tacka sa najvecim prosecnim rastojanjem do ostalih
        avg_d = [sum(euclidean(data[i], data[j]) for j in cluster if j != i) /
                 max(1, len(cluster) - 1) for i in cluster]
        seed_idx = cluster[avg_d.index(max(avg_d))]
        splinter = [seed_idx]
        rest = [i for i in cluster if i != seed_idx]
        changed = True
        while changed:
            changed = False
            for i in list(rest):
                d_spl = sum(euclidean(data[i], data[j]) for j in splinter) / len(splinter)
                d_rest = (sum(euclidean(data[i], data[j]) for j in rest if j != i) /
                          max(1, len(rest) - 1))
                if d_rest > d_spl:
                    rest.remove(i); splinter.append(i); changed = True
        clusters = [c for kk, c in enumerate(clusters) if kk != best_ci] + [splinter, rest]
        step += 1
        history.append(([c[:] for c in clusters],
                        f"Korak {step}: podeljen klaster (dijam={best_diam:.2f}) -> {len(clusters)} klastera"))
    return history

def next_diana(data, history, title="DIANA (top-down)", reset=False):
    _step("diana", data, history, draw_clusters_2d, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_diana(data, diana_history, reset=True)
diana_history = diana_steps(data, target_k=3)
next_diana(data, diana_history)


### DBSCAN

Za svaku tacku gleda eps-okolinu. Ako u njoj ima bar `min_pts` suseda, tacka je **jezgro** i pokrece/prosiruje klaster; inace je **šum**. Ima 27 koraka (jedan po obradjenoj tacki).

**Opis:**
- **Sive kruznice**: tacke koje algoritam jos nije obradio
- **Obojene kruznice (plava/narandzasta/zelena)**: tacke dodeljene klasterima 1/2/3
- **Crni "x"**: tacke obelezene kao **šum**
- **Crveni krugovi bez ispune**: tacka koja se trenutno obradjuje + njene tacke u eps-okolini
- **Naslov panela**: indeks tacke u obradi, broj suseda, i odluka (jezgro / šum / granicna)
- **Sta gledati**: kako se klaster siri "talasno" iz jezgrenih tacaka; tri šum tacke u sredini (oko (3,3), (3,2), (4,1.5)) ce ostati kao crni `x` jer su preudaljene od svake guste grupe

In [ ]:
def dbscan_steps(data, eps, min_pts):
    """DBSCAN; vraca [(labels, current_indices, opis), ...].
    labels: 0 = neobelezeno, -1 = sum, 1.. = ID klastera."""
    n = len(data)
    labels = [0] * n
    history = []
    cluster_id = 0
    for i in range(n):
        if labels[i] != 0:
            continue
        N = neighbors(data, i, eps)
        if len(N) < min_pts:
            labels[i] = -1
            history.append((labels[:], [i],
                            f"Tacka {i}: {len(N)} suseda < {min_pts} -> sum"))
            continue
        cluster_id += 1
        labels[i] = cluster_id
        seed = list(N)
        history.append((labels[:], [i] + seed,
                        f"Tacka {i}: jezgro ({len(N)} suseda) "
                        f"-> novi klaster {cluster_id}"))
        idx = 0
        while idx < len(seed):
            j = seed[idx]
            idx += 1
            if labels[j] == -1:
                labels[j] = cluster_id
                continue
            if labels[j] != 0:
                continue
            labels[j] = cluster_id
            Nj = neighbors(data, j, eps)
            if len(Nj) >= min_pts:
                fresh = [x for x in Nj if x not in seed]
                seed.extend(fresh)
                history.append((labels[:], [j] + Nj,
                                f"Tacka {j}: jezgro u klasteru {cluster_id}, "
                                f"siri se za {len(fresh)} novih"))
            else:
                history.append((labels[:], [j],
                                f"Tacka {j}: granicna u klasteru {cluster_id}"))
    return history

def draw_dbscan_step(ax, data, history, idx, title):
    labels, current, opis = history[idx]
    for i, p in enumerate(data):
        l = labels[i]
        if l == 0:
            ax.scatter([p[0]], [p[1]], c="lightgray", s=95, edgecolors="black", linewidths=0.5)
        elif l == -1:
            ax.scatter([p[0]], [p[1]], c="black", marker="x", s=90, linewidths=2)
        else:
            ax.scatter([p[0]], [p[1]],
                       c=COLORS[(l - 1) % len(COLORS)], s=95, edgecolors="black", linewidths=0.5)
    for i in current:
        ax.scatter([data[i][0]], [data[i][1]],
                   facecolors="none", edgecolors="red",
                   s=240, linewidths=2)
    ax.set_title(f"{title} (korak {idx+1}/{len(history)})\n{opis}",
                 fontsize=10)

def next_dbscan(data, history, title="DBSCAN", reset=False):
    _step("db", data, history, draw_dbscan_step, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_dbscan(data, db_history, reset=True)
db_history = dbscan_steps(data, eps=1.5, min_pts=2)
next_dbscan(data, db_history)


### CAST (Cluster Affinity Search Technique)

Gradi klaster po klaster: dodaje spoljnu tacku sa najvecom prosecnom slicnoscu ka klasteru (ako > theta), uklanja unutrasnju tacku sa najnizom (ako < theta). Originalno koristi Pearson korelaciju; ovde je za 2D slicnost = $1 - d/d_{max}$.

**Opis:**
- **Sive tacke**: jos nedodeljene tacke (cekaju na red)
- **Obojene tacke**: tacke vec dodeljene zatvorenim klasterima
- **Crveni krugovi bez ispune**: tacke u **trenutno aktivnom** klasteru koji se gradi (add/remove faza)
- **Naslov panela**: ID aktivnog klastera + akcija u koraku (`dodata tacka X` ili `uklonjena tacka X` sa vrednoscu slicnosti)
- **Sta gledati**: algoritam ne treba k unapred - vidi se kako automatski otvara nov klaster kad iscrpi okolinu prethodnog; theta = 0.6 znaci da samo dovoljno slicne tacke ulaze

In [ ]:
def cast_steps(data, theta=0.6):
    n = len(data)
    max_d = max(euclidean(data[i], data[j]) for i in range(n) for j in range(i + 1, n))
    sim = [[1 - euclidean(data[i], data[j]) / max_d if i != j else 1.0
            for j in range(n)] for i in range(n)]
    labels = [0] * n
    cluster_id = 0
    history = []
    unassigned = list(range(n))
    while unassigned:
        cluster_id += 1
        seed = unassigned[0]
        cluster = [seed]; labels[seed] = cluster_id
        history.append((labels[:], cluster[:],
                        f"Novi klaster {cluster_id}, seed tacka {seed}"))
        changed = True
        iter_cap = 0
        while changed and iter_cap < 200:
            iter_cap += 1
            changed = False
            # ADD
            best, best_s = None, theta
            for i in unassigned:
                if i in cluster: continue
                s = sum(sim[i][j] for j in cluster) / len(cluster)
                if s > best_s: best_s, best = s, i
            if best is not None:
                cluster.append(best); labels[best] = cluster_id
                history.append((labels[:], cluster[:],
                                f"Klaster {cluster_id}: dodata tacka {best} (sl={best_s:.2f})"))
                changed = True; continue
            # REMOVE
            worst, worst_s = None, theta
            if len(cluster) > 1:
                for i in list(cluster):
                    s = sum(sim[i][j] for j in cluster if j != i) / (len(cluster) - 1)
                    if s < worst_s: worst_s, worst = s, i
            if worst is not None:
                cluster.remove(worst); labels[worst] = 0
                history.append((labels[:], cluster[:],
                                f"Klaster {cluster_id}: uklonjena tacka {worst} (sl={worst_s:.2f})"))
                changed = True
        unassigned = [i for i in range(n) if labels[i] == 0]
    return history

def draw_cast_step(ax, data, history, idx, title):
    labels, current, opis = history[idx]
    for i, p in enumerate(data):
        l = labels[i]
        if l == 0:
            ax.scatter([p[0]], [p[1]], c="lightgray", s=95, edgecolors="black", linewidths=0.5)
        else:
            ax.scatter([p[0]], [p[1]], c=COLORS[(l - 1) % len(COLORS)], s=95, alpha=0.9, edgecolors="black", linewidths=0.5)
    for i in current:
        ax.scatter([data[i][0]], [data[i][1]],
                   facecolors="none", edgecolors="red", s=240, linewidths=2)
    ax.set_title(f"{title} (korak {idx+1}/{len(history)})\n{opis}", fontsize=10)

def next_cast(data, history, title="CAST (theta=0.6)", reset=False):
    _step("cast", data, history, draw_cast_step, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_cast(data, cast_history, reset=True)
cast_history = cast_steps(data, theta=0.6)
next_cast(data, cast_history)


### Louvain (i Leiden) - community detection na grafu

Ne radi na koordinatama nego na **grafu**. Prvo izgradimo graf blizine: cvor = tacka, grana ako $d \leq \varepsilon$, tezina $w = 1/(d+0.01)$. Onda Louvain pohlepno prebacuje cvor u zajednicu suseda ako to poveca modularnost $Q$.

**Opis:**
- **Sive linije** (tanke, ispod cvorova): grane grafa - vidljive su gusto unutar prirodnih grupa, retko izmedju
- **Sive kruznice**: cvorovi koji su **singletoni** (sami u svojoj zajednici) ili u sitnim grupicama
- **Obojene kruznice sa crnim obrubom**: cvorovi u top-6 najvecih zajednica (boja = ID zajednice)
- **Korak 0**: 27 sivih cvorova (svaki je svoja zajednica)
- **Naslov panela**: koji cvor je prebacen u koju zajednicu + ukupno `[X klastera + Y singletona]`
- **Sta gledati**: kako se sivi cvorovi postepeno pretvaraju u obojene kako prave zajednice rastu spajanjem; finalna podela odrazava gust podgraf strukturu

**Leiden** je poboljsanje Louvaina: dodaje refinement fazu koja sprecava da nastanu lose povezane (raskomadane) zajednice. Vizualizacija koraka je sustinski ista, samo sa dodatnom proverom posle Louvain faze.

In [ ]:
LOUVAIN_ADJ = None  # graf cuvamo u modulu da ga draw funkcija vidi

def louvain_steps(data, eps=1.5):
    global LOUVAIN_ADJ
    n = len(data)
    adj = {i: {} for i in range(n)}
    for i in range(n):
        for j in range(i + 1, n):
            d = euclidean(data[i], data[j])
            if d <= eps:
                w = 1.0 / (d + 0.01)
                adj[i][j] = w; adj[j][i] = w
    LOUVAIN_ADJ = adj
    m = sum(sum(v.values()) for v in adj.values()) / 2.0
    deg = {i: sum(adj[i].values()) for i in range(n)}
    community = list(range(n))
    history = [(community[:], "Korak 0: svaka tacka je svoja zajednica")]
    step = 0
    improved = True
    iter_cap = 0
    while improved and iter_cap < 50:
        iter_cap += 1
        improved = False
        for i in range(n):
            best_c, best_gain = community[i], 0.0
            for c in set(community[j] for j in adj[i]):
                if c == community[i]: continue
                k_in = sum(adj[i].get(j, 0) for j in range(n) if community[j] == c)
                sum_deg_c = sum(deg[j] for j in range(n) if community[j] == c)
                gain = k_in / m - deg[i] * sum_deg_c / (2 * m * m) if m > 0 else 0
                if gain > best_gain: best_gain, best_c = gain, c
            if best_c != community[i]:
                old = community[i]; community[i] = best_c
                step += 1
                history.append((community[:],
                                f"Korak {step}: cvor {i} prebacen iz zajednice {old} u {best_c}"))
                improved = True
                if step >= 30:
                    return history
    return history

def draw_louvain_step(ax, data, history, idx, title):
    community, opis = history[idx]
    # grane prvo (ispod cvorova)
    if LOUVAIN_ADJ is not None:
        for i in range(len(data)):
            for j in LOUVAIN_ADJ[i]:
                if j > i:
                    ax.plot([data[i][0], data[j][0]], [data[i][1], data[j][1]],
                            c="lightgray", linewidth=0.6, zorder=1)
    # Boja samo top-N najvecih zajednica; ostali sivi.
    # Razlog: na pocetku Louvain ima 27 singletona; ciklanje 6 boja
    # bi izgledalo kao sum. Ovako se vidi NASTAJANJE pravih klastera.
    sizes = {}
    for c in community:
        sizes[c] = sizes.get(c, 0) + 1
    # ranguj zajednice po velicini, samo one sa >=2 clana se boje
    real = sorted((c for c, n in sizes.items() if n >= 2),
                  key=lambda c: -sizes[c])[:len(COLORS)]
    color_map = {c: COLORS[i] for i, c in enumerate(real)}
    for i, p in enumerate(data):
        c = community[i]
        if c in color_map:
            ax.scatter([p[0]], [p[1]], c=color_map[c], s=95, zorder=2,
                       edgecolors="black", linewidths=0.6)
        else:
            ax.scatter([p[0]], [p[1]], c="lightgray", s=60, zorder=2,
                       edgecolors="gray", linewidths=0.4)
    n_real = len(real)
    n_single = sum(1 for n in sizes.values() if n == 1)
    ax.set_title(f"{title} (korak {idx+1}/{len(history)})\n"
                 f"{opis}  [{n_real} klastera + {n_single} singletona]",
                 fontsize=10)

def next_louvain(data, history, title="Louvain", reset=False):
    _step("louv", data, history, draw_louvain_step, title, reset)

In [ ]:
# Izgradnja koraka za ovaj algoritam, pa Shift+Enter (ponovo i ponovo) za sledeci korak.
# Za pocetak iz koraka 1: next_louvain(data, louvain_history, reset=True)
louvain_history = louvain_steps(data, eps=1.5)
next_louvain(data, louvain_history)


## Više vizualizacija

- **K-means:** [naftaliharris.com/blog/visualizing-k-means-clustering](https://www.naftaliharris.com/blog/visualizing-k-means-clustering/)
- **DBSCAN:** [naftaliharris.com/blog/visualizing-dbscan-clustering](https://www.naftaliharris.com/blog/visualizing-dbscan-clustering/)